# Redis Key Schema

This notebook walks through the Redis schema used by the updated synthetic MAID demo.

The important change is that requests no longer start from a direct user key. They start from an identity token.


## Key Families

The serving path uses three main Redis object families:

- `identity:<token>` -> MAID id
- `maid:<id>` -> full MAID profile hash
- `campaign:<id>` -> campaign hash
- `idx:*` -> set-based inverted indexes


In [1]:
key_examples = {
    'identity': 'identity:id_00001_01',
    'maid': 'maid:maid_00001',
    'campaign': 'campaign:c00042',
    'indexes': [
        'idx:card_tier:Platinum',
        'idx:geo:US',
        'idx:device_type:mobile',
        'idx:device:iOS',
        'idx:segment:travel_high',
    ],
}
key_examples


{'identity': 'identity:id_00001_01',
 'maid': 'maid:maid_00001',
 'campaign': 'campaign:c00042',
 'indexes': ['idx:card_tier:Platinum',
  'idx:geo:US',
  'idx:device_type:mobile',
  'idx:device:iOS',
  'idx:segment:travel_high']}

## Index Dimensions

Candidate generation currently indexes:

- card tier
- country
- device type
- device OS
- strong user segments

Campaigns that target all values on a dimension are expanded into all concrete index sets during load. That keeps the request path simple and avoids wildcard logic in the query itself.


In [2]:
lookup_groups = [
    ['idx:card_tier:Platinum', 'idx:geo:US', 'idx:device_type:mobile', 'idx:device:iOS', 'idx:segment:travel_high'],
    ['idx:card_tier:Platinum', 'idx:geo:US', 'idx:device_type:mobile', 'idx:segment:travel_high'],
    ['idx:card_tier:Platinum', 'idx:geo:US'],
    ['idx:geo:US', 'idx:segment:travel_high'],
]
['SINTER ' + ' '.join(group) for group in lookup_groups]


['SINTER idx:card_tier:Platinum idx:geo:US idx:device_type:mobile idx:device:iOS idx:segment:travel_high',
 'SINTER idx:card_tier:Platinum idx:geo:US idx:device_type:mobile idx:segment:travel_high',
 'SINTER idx:card_tier:Platinum idx:geo:US',
 'SINTER idx:geo:US idx:segment:travel_high']

## Request Flow

The high-level request path is:

1. `GET identity:<token>`
2. `HGETALL maid:<id>`
3. one or more `SINTER` probes across the index sets
4. `HGETALL campaign:<id>` in a pipeline batch
5. exact filtering in the app for:
   - state
   - postal code
   - pacing
   - frequency cap
6. reranking in memory


In [3]:
flow = {
    'step_1': 'GET identity:id_00001_01',
    'step_2': 'HGETALL maid:maid_00001',
    'step_3': 'SINTER idx:card_tier:Platinum idx:geo:US idx:device_type:mobile idx:device:iOS idx:segment:travel_high',
    'step_4': 'pipeline(HGETALL campaign:c00042, ...)',
    'step_5': 'exact eligibility checks in Python',
    'step_6': 'linear reranking',
}
flow


{'step_1': 'GET identity:id_00001_01',
 'step_2': 'HGETALL maid:maid_00001',
 'step_3': 'SINTER idx:card_tier:Platinum idx:geo:US idx:device_type:mobile idx:device:iOS idx:segment:travel_high',
 'step_4': 'pipeline(HGETALL campaign:c00042, ...)',
 'step_5': 'exact eligibility checks in Python',
 'step_6': 'linear reranking'}

## Why Some Filters Are Indexed And Others Are App-Side

The demo indexes the fields that are small, reusable, and selective enough to help candidate generation.

The remaining filters are applied after campaign materialization because they are either:

- higher-cardinality
- more stateful
- or better treated as exact checks than as primary retrieval indexes

That currently includes state, postal code, pacing, and frequency cap.
